<a href="https://colab.research.google.com/github/putrinayshila/KKAXI_EDA/blob/main/KKAEDA_kelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**- Nama Anggota Kelompok :**
1. Christabel Yemima Agatha (09)
2. Putri Nayshila Intan Maharani (28)

**- Dataset yang dipilih**
(dataset_nilai_akademik_siswa)

**- Pertanyaan Analisis Awal :**
1. Mata pelajaran mana yang memiliki rata-rata nilai tertinggi dan terendah?

2. Bagaimana perbandingan rata-rata nilai siswa pada Ujian Harian (UH), Uts (UTS), dan Uas (UAS)?

3. Kelas mana (misalnya XI RPL 1, XI RPL 2, dst.) yang memiliki persentase ketuntasan tertinggi (misal KKM = 75)?

**- Dugaan Masalah
Kualitas Data :**

1. Ada data anomali/outlier

2. Missing Value pada kolom

3. Data yang duplikat

4. Format data tidak sesuai

**- Rencana Teknik
Pembersihan :**

1. Menyamakan format teks yang tidak seragam

2. Menghapus kata 'poin' pada kolom nilai agar bisa mengubah tipe data menjadi integer

3. Menghapus baris data yang duplikat

**- Rencana Manipulasi
Data :**

**Filter:**  Saring siswa di bawah KKM (nilai < 75) atau berdasarkan jenis_ujian == 'UAS'

**Sort:** Urutkan data berdasarkan nilai dari tertinggi ke terendah (descending).

**Kolom turunan:** Buat kolom status_kelulusan ('Tuntas' jika $\ge 75$, 'Remedial' jika $< 75$).

**Groupby/agregasi:** Hitung rata-rata nilai per mata_pelajaran dan persentase ketuntasan per kelas.

**- Jadwal Kerja :**
1. 18.00 – 18.30 (Maghrib Selesai): Data Loading & InspectionBuka file, cek missing value, temukan teks 'poin', dan tandai nilai outlier 999.   
2. 18.30 – 19.15: Data CleaningBuang kata 'poin', hapus nilai 999 & NaN pada nilai.   Samakan kapitalisasi jenis_ujian (UH, UTS, UAS) dan format tanggal.   
3. 19.15 – 20.37: Data Manipulation & ExportBuat kolom status_kelulusan ('Tuntas' / 'Remedial').   Lakukan filter, sort, dan groupby agregasi rata-rata nilai.   Export/Simpan file sebagai dataset_bersih.csv.   
4. 19.45 – 21.15: Profiling & RefleksiTulis 2–3 poin narasi temuan utama dari hasil olah data.   Isi lembar refleksi proyek (B.7).   
5. 21.15 – 20.30: Final CheckCek ulang kelengkapan tugas sebelum dikumpulkan/ditutup.

**- Pembagian Peran :**

Pembagian Peran Kelompok
Anggota 1:
- Teknis Coding (Sel 4 & Sel 5)
- Evaluasi Proyek

Anggota 2:
- Teknis Coding (Sel 1 s/d Sel 3)   
- Analisis Data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Alur Ke-2
# Mengimpor file dataset yang sudah di-upload
df = pd.read_csv('dataset_nilai_akademik_siswa - dataset_nilai_akademik_siswa (1).csv')

# Cek 5 data teratas untuk memastikan data berhasil diimpor
df.head()

,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
0,SIS0004,Joko Prasetyo,XI RPL 2,KKA,uas,06/08/2026,46,Ibu Wati
1,SIS0020,Eka Putri,XI RPL 1,PKK,uts,12 Agustus 2026,73,Bpk. Santoso
2,SIS0015,Nanda Pratama,XI RPL 3,Pemrograman Web,UAS,2026-08-15,69,Bpk. Santoso
3,SIS0046,Fajar Nugroho,XI RPL 2,Matematika,UH,10/08/2026,73,Bpk. Arifin
4,SIS0011,Ayu Lestari,XI RPL 2,Pemrograman Web,UH,6 Agustus 2026,53,Ibu Wati


In [ ]:
# 1. Menampilkan tipe data dan jumlah missing value
print("--- Ringkasan Tipe Data & Missing Value ---")
print(df.info())

# 2. Menampilkan statistik deskriptif awal
print("\n--- Statistik Deskriptif ---")
display(df.describe(include='all'))

# 3. Menampilkan ukuran data
print(f"\nUkuran Dataset: {df.shape[0]} Baris, {df.shape[1]} Kolom")

--- Ringkasan Tipe Data & Missing Value ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_siswa        79 non-null     object
 1   nama            79 non-null     object
 2   kelas           79 non-null     object
 3   mata_pelajaran  79 non-null     object
 4   jenis_ujian     79 non-null     object
 5   tanggal_ujian   79 non-null     object
 6   nilai           75 non-null     object
 7   guru_pengampu   75 non-null     object
dtypes: object(8)
memory usage: 5.1+ KB
None

--- Statistik Deskriptif ---


,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
count,79,79,79,79,79,79,75,75
unique,75,20,3,6,6,38,47,5
top,SIS0046,Lukman Hakim,XI RPL 3,PKK,UH,2026-08-05,73,Bpk. Arifin
freq,2,7,34,16,21,8,5,18



Ukuran Dataset: 79 Baris, 8 Kolom


In [ ]:
# 1. Bersihkan teks 'poin' pada kolom nilai & ubah tipe data ke numerik
df['nilai_clean'] = df['nilai'].astype(str).str.replace('poin', '').str.strip()
df['nilai_clean'] = pd.to_numeric(df['nilai_clean'], errors='coerce')

# 2. Hapus nilai anomali/outlier (>100) dan missing value (NaN) pada kolom nilai
df_clean = df[(df['nilai_clean'].notnull()) & (df['nilai_clean'] <= 100)].copy()
df_clean['nilai'] = df_clean['nilai_clean'].astype(int)
df_clean.drop(columns=['nilai_clean'], inplace=True)

# 3. Isi missing value pada kolom guru_pengampu
df_clean['guru_pengampu'] = df_clean['guru_pengampu'].fillna('Belum Terdata')

# 4. Seragamkan format jenis_ujian menjadi huruf kapital
df_clean['jenis_ujian'] = df_clean['jenis_ujian'].str.upper()

# 5. Seragamkan format tanggal_ujian ke YYYY-MM-DD
df_clean['tanggal_ujian'] = pd.to_datetime(df_clean['tanggal_ujian'], errors='coerce').dt.strftime('%Y-%m-%d')

# 6. Hapus baris data yang duplikat
df_clean = df_clean.drop_duplicates()

print("Proses Pembersihan Data Selesai!")
print(f"Jumlah baris setelah dibersihkan: {df_clean.shape[0]} baris")

Proses Pembersihan Data Selesai!
Jumlah baris setelah dibersihkan: 70 baris


In [ ]:
# Alur ke-3
import numpy as np

In [ ]:
df_clean['status_kelulusan'] = np.where(df_clean['nilai'] >= 75, 'Tuntas', 'Remedial')

In [ ]:
df_sorted = df_clean.sort_values(by='nilai', ascending=False)

In [ ]:
df_remedial = df_clean[df_clean['nilai'] < 75]
df_uas = df_clean[df_clean['jenis_ujian'] == 'UAS']

In [ ]:
avg_mapel = df_clean.groupby('mata_pelajaran')['nilai'].mean().round(2).reset_index()

In [ ]:
avg_ujian = (
    df_clean.groupby('jenis_ujian')['nilai']
    .mean()
    .round(2)
    .reset_index()
)
display(avg_ujian)

,jenis_ujian,nilai
0,UAS,60.84
1,UH,70.35
2,UTS,60.68


In [ ]:
display(avg_mapel)

,mata_pelajaran,nilai
0,Bahasa Inggris,67.17
1,Basis Data,59.50
2,KKA,64.25
3,Matematika,66.09
4,PKK,63.31
5,Pemrograman Web,67.31


In [ ]:
ketuntasan = (
    df_clean.groupby('kelas')['status_kelulusan']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    * 100
).round(2)
display(ketuntasan)

status_kelulusan,Remedial,Tuntas
kelas,,
XI RPL 1,70.59,29.41
XI RPL 2,71.43,28.57
XI RPL 3,90.62,9.38


In [ ]:
df_clean.to_csv('dataset_bersih.csv', index=False)

**Hasil Analisis:**
1. Mata Pelajaran dengan Nilai Tertinggi & Terendah:
- Tertinggi: Pemrograman Web (67.31) dan Bahasa Inggris (67.2).  

- Terendah: Basis Data (59.5) dan PKK (63.31).   

 - Kesimpulan: Siswa mengalami kendala paling besar pada mata pelajaran Basis Data.

2. Perbandingan Nilai Ujian Harian (UH), UTS, dan UAS:

Ujian Harian (UH): 70.35

UTS: 60.68

UAS: 60.84

Kesimpulan: Rata-rata nilai Ujian Harian (UH) merupakan yang tertinggi (70.35). Namun, nilai siswa mengalami penurunan saat menghadapi ujian besar seperti UTS (60.68) dan UAS (60.84).

3. Persentase Ketuntasan per Kelas (KKM = 75):
- XI RPL 1: 29.41% Tuntas dan 70.59% Remedial   
- XI RPL 2: 28.57% Tuntas dan 71.43% Remedial   
- XI RPL 3: 9.38% Tuntas dan 90.62% Remedial   
- Kesimpulan: XI RPL 1 memiliki persentase ketuntasan tertinggi (29.41%), sedangkan XI RPL 3 paling rendah (9.38%). Secara keseluruhan, lebih dari 70% siswa di setiap kelas masih perlu mengikuti remedial.

**Lembar Refleksi Proyek**

6. Tahap mana (loading, inspection, cleaning, atau manipulation) yang paling menantang bagi kelompokmu, dan bagaimana kalian mengatasinya?

Jawaban = Bagian yang paling menantang bagi kelompok saya adalah saat proses manipulasi data karena ingin menjaga data agar tetap valid. Mengisi atau menghapus data sembarangan bisa membuat hasil analisis bias dan membuang informasi penting, jadi setiap keputusan pembersihan harus punya alasan yang jelas.

7. Menurutmu, mengapa keputusan membersihkan data (mis. menghapus vs mengisi data kosong) perlu didasarkan pada alasan yang jelas, bukan asal-asalan?

Jawaban = Mencegah Bias dan menjaga Validitas data karena jika asal mengisi  data kosong dan menghapus bisa merusak nilai rata-rata dan mengurangi resiko terhapusnya data yang penting.

8. Apa hubungan antara dataset bersih hasil proyek ini dengan pekerjaan seorang Data Analyst di dunia nyata?

Jawaban = Karena dengan data yang bersih keputusan yang akan diambil berdasarkan data yang ada akan valid, namu jika menggunakan data yang masih kotor bisa terjadi kesalahan strategi yang berujung pada kesalahan keputusan.